# Caso E · 03 Features meteorológicos para Caso B

> _Tutorial · Caso de uso: **E — Meteo & solar** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Construir features meteorológicas reusables por el modelo de consumo del Caso B.


## 2. Qué se aprende

- Lag/lead temporal de variables exógenas.
- Dewpoint, sensación térmica.
- Diurnal/seasonal encoding.


## 3. Contexto del caso de uso

Caso B necesita `t_outdoor`, `ghi`, `dewpoint`, `is_daylight`.


## 4. Relación con CENTINELA+

Features se calculan al vuelo en producción.


## 5. Relación con Medallion

Lee plata, escribe oro.


## 6. Datos de entrada

Mock ERA5.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Cargamos y derivamos.


In [2]:
csv_path = ROOT / "notebooks" / "_data" / "era5_xativa_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"]).set_index("timestamp")

def magnus_dewpoint(t, rh):
    a, b = 17.625, 243.04
    rh = np.clip(rh, 1, 100)
    alpha = np.log(rh / 100.0) + (a * t) / (b + t)
    return (b * alpha) / (a - alpha)

# RH no está en el mock; aproximamos con anti-correlación de T
rh_mock = np.clip(70 - (df["t_air_c"] - 18) * 1.5, 30, 90)
df["dewpoint_c"] = magnus_dewpoint(df["t_air_c"], rh_mock)
df["is_daylight"] = (df["ghi_w_m2"] > 50).astype(int)
df.head()


,t_air_c,ghi_w_m2,wind_speed_ms,precip_mm,pressure_hpa,dewpoint_c,is_daylight
timestamp,,,,,,,
2024-06-01 00:00:00+00:00,19.82,0.0,1.00,0.00,1016.7,13.576937,0
2024-06-01 01:00:00+00:00,18.68,0.0,2.54,0.00,1015.6,12.873547,0
2024-06-01 02:00:00+00:00,21.06,0.0,0.00,0.01,1015.6,14.324305,0
2024-06-01 03:00:00+00:00,22.21,0.0,4.16,0.00,1014.5,15.000166,0
2024-06-01 04:00:00+00:00,20.56,0.0,1.78,0.01,1018.8,14.025221,0


## 10. Exploración paso a paso

Estadísticas.


In [3]:
print(df[["t_air_c", "ghi_w_m2", "dewpoint_c", "is_daylight"]].describe().round(2))


       t_air_c  ghi_w_m2  dewpoint_c  is_daylight
count   720.00    720.00      720.00       720.00
mean     25.83    228.88       16.86         0.46
std       4.40    302.46        2.32         0.50
min      17.02      0.00       11.82         0.00
25%      21.73      0.00       14.72         0.00
50%      25.77      0.00       16.98         0.00
75%      29.94    475.00       19.05         1.00
max      33.78    950.00       20.67         1.00


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Persistimos.


In [4]:
out_dir = ROOT / "output" / "case_E"
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / "weather_features.parquet"
df.to_parquet(parquet_path)
print(f"Wrote {parquet_path.relative_to(ROOT)}")


Wrote output\case_E\weather_features.parquet


## 13. Visualizaciones explicativas

Dewpoint vs T_air.


In [5]:
plt.figure(figsize=(7, 3))
plt.scatter(df["t_air_c"], df["dewpoint_c"], s=4, color="#3F51B5")
plt.plot([0, 35], [0, 35], "--", color="gray")
plt.xlabel("T air (°C)"); plt.ylabel("Dewpoint (°C)")
plt.title("Dewpoint sigue a T_air pero queda por debajo")
plt.tight_layout()


## 14. Validaciones

Dewpoint <= T_air siempre.


In [6]:
assert (df["dewpoint_c"] <= df["t_air_c"] + 0.5).all()
print("Sanity check OK")


Sanity check OK


## 15. Errores comunes

1. Usar RH > 100 sin clip.
2. Aplicar Magnus a T en K.
3. Promediar `is_daylight` (categórica).


## 16. Ejercicios propuestos

1. Añade `solar_zenith_angle`.
2. Calcula `solar_irradiance_clear_sky` y compara.
3. Estima `wind_chill`.


## 17. Cómo se reutiliza con datos reales

Idéntico — solo cambia origen.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `05_case_E_weather_solar/04_prediccion_solar.ipynb`.
- Documento web del caso: `docs/use-cases/case-e-weather-solar.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo de irradiancia solar global (Iqbal 1983)

$$
G_h(t) = G_b(t) + G_d(t), \quad G_b(t) = G_{sc} \cdot \tau_b(t) \cdot \cos\theta_z(t)
$$

con $G_{sc} = 1361$ W/m² constante solar y $\theta_z$ ángulo cenital del sol:

$$
\cos\theta_z = \sin\delta \sin\phi + \cos\delta \cos\phi \cos\omega
$$

donde $\delta$ es declinación solar, $\phi$ latitud (Xátiva 38.99°N) y
$\omega$ ángulo horario.

### Clear-sky index

$$
k_c(t) = \frac{G_h(t)}{G_{clear}(t)} \in [0, 1]
$$

separa astronomía (determinista) de meteorología (estocástica).

### Predictor XGBoost para FV

$$
\hat{P}(t+h) = P_{nominal} \cdot \eta_{panel} \cdot \text{XGB}(k_c(t), T_{out}, t_{hora}, t_{año})
$$

### Métrica Skill Score

$$
\text{Skill} = 1 - \frac{\text{RMSE}_{model}}{\text{RMSE}_{persistence}}
$$

Objetivo Simarro: $\text{nMAE} \leq 8\%$ a 24 h, $\text{Skill} \geq 0.3$.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Predicción solar permite optimizar despacho energético en centros con FV instalada y planificar climatización aprovechando picos de radiación. CAPTIA puede ofrecer este caso como **producto añadido** a centros con paneles.

### ROI estimado

| Concepto | Valor |
|---|---|
| Optimización despacho FV (centro 50 kWp) | +800 €/año |
| Sinergia con Caso B forecast | +500 €/año |
| Coste integración ERA5+AEMET | -1 200 € one-time |
| **Payback** | **~12 meses** |


## 21. Bibliografía y referencias

- Iqbal, M. (1983). *An Introduction to Solar Radiation*. Academic Press.
- ECMWF (2024). *ERA5 Reanalysis Documentation*. Copernicus Climate Change Service.
- AEMET. *Open Data Portal*. https://opendata.aemet.es
- Holmgren, W. F. et al. (2018). *pvlib python: a python package for modeling solar energy systems*. JOSS 3(29).
